In [1]:
import pandas as pd
import os
from collections import Counter

In [2]:
def sentence_to_list(sentence):
    word_list = []
    word = ''
    for c in str(sentence).lower():
        if c.isalpha():
            word += c
        else:
            if word: word_list.append(word)
            word = ''
    if word: word_list.append(word)
    return word_list

In [3]:
def run_delta_analysis(dir_original, dir_corrected, output_report):
    changes = []
    files = [f for f in os.listdir(dir_original) if f.endswith('.csv')]
    
    print(f"Comparing {len(files)} files between directories...")
    
    for file in files:
        path_orig = os.path.join(dir_original, file)
        path_corr = os.path.join(dir_corrected, file)
        
        if not os.path.exists(path_corr):
            print(f"Missing corrected file for {file}")
            continue
            
        df_orig = pd.read_csv(path_orig)
        df_corr = pd.read_csv(path_corr)
        
        # Ensure row counts match before comparing
        if len(df_orig) != len(df_corr):
            print(f"Row count mismatch in {file}. Skipping.")
            continue
            
        for idx in range(len(df_orig)):
            # Compare Source Column (kalimat_asal)
            orig_asal_words = sentence_to_list(df_orig.loc[idx, 'kalimat_asal'])
            corr_asal_words = sentence_to_list(df_corr.loc[idx, 'kalimat_asal'])
            
            if len(orig_asal_words) == len(corr_asal_words):
                for w_o, w_c in zip(orig_asal_words, corr_asal_words):
                    if w_o != w_c:
                        changes.append((w_o, w_c, file))
                        
            # Compare Target Column (kalimat_tujuan)
            orig_tujuan_words = sentence_to_list(df_orig.loc[idx, 'kalimat_tujuan'])
            corr_tujuan_words = sentence_to_list(df_corr.loc[idx, 'kalimat_tujuan'])
            
            if len(orig_tujuan_words) == len(corr_tujuan_words):
                for w_o, w_c in zip(orig_tujuan_words, corr_tujuan_words):
                    if w_o != w_c:
                        changes.append((w_o, w_c, file))
                        
    # Aggregate and count all unique changes
    change_counts = Counter([(o, c) for o, c, f in changes])
    
    report_data = []
    for (orig, corr), count in change_counts.most_common():
        # Identify which dictionary files contain this specific error
        files_affected = list(set([f for o, c, f in changes if o == orig and c == corr]))
        file_string = ", ".join(files_affected[:3]) + ("..." if len(files_affected) > 3 else "")
        
        report_data.append({
            'Original_Word': orig,
            'Corrected_Word': corr,
            'Frequency': count,
            'Sample_Files': file_string
        })
        
    report_df = pd.DataFrame(report_data)
    report_df.to_csv(output_report, index=False)
    
    print(f"Analysis complete. Found {len(report_df)} unique spelling corrections.")
    return report_df

# Execute the crosscheck
dir_11 = '../Ekstraksi/11. Parallel Corpus'
dir_12 = '../Ekstraksi/12. Parallel Corpus - Spelling Checker'
report_path = '../csvAnalysis/Spelling_Delta_Report.csv'

report_df = run_delta_analysis(dir_11, dir_12, report_path)
display(report_df.head(20))

Comparing 60 files between directories...
Analysis complete. Found 82483 unique spelling corrections.


,Original_Word,Corrected_Word,Frequency,Sample_Files
0,menanamnya,menantunya,1,10_Parcor.csv
1,ngehambur,ngahambur,1,10_Parcor.csv
2,sabalupm,sabalun,1,10_Parcor.csv
3,faran,jaran,1,10_Parcor.csv
4,luju,tuju,1,10_Parcor.csv
5,fangil,langil,1,10_Parcor.csv
6,keadaatn,keadaan,1,10_Parcor.csv
7,mangkitn,mungkin,1,10_Parcor.csv
8,menurlll,menurul,1,10_Parcor.csv
9,fakilah,iakilah,1,10_Parcor.csv
